In [28]:
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import MNIST

In [45]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

trainset=MNIST(root="./MNdata",train=True,download=True,transform=transform)
testset=MNIST(root="./MNdata",train=False,download=True,transform=transform)

In [47]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./MNdata
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [48]:
trainloader=DataLoader(trainset,batch_size=32,shuffle=True)
testloader=DataLoader(testset,batch_size=32)

In [55]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(7 * 7 * 64, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

 
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc_layers(x)
        return x

In [57]:
import torch.optim as optim
model=CNN()
criteria=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters())

In [59]:
train_losses = []
val_losses = []

epochs = 10  # 100 is fine but start small for debugging

for epoch in range(epochs):
    # ---- TRAINING ----
    model.train()
    running_loss = 0.0 
    
    for images, labels in trainloader:
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criteria(outputs, labels)  # fixed spelling
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    epoch_train_loss = running_loss / len(trainloader)  # fixed name
    train_losses.append(epoch_train_loss)

    # ---- VALIDATION ----
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in testloader:
            outputs = model(images)
            loss = criteria(outputs, labels)
            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(testloader)  # fixed name
    val_losses.append(epoch_val_loss)

    print(f"Epoch {epoch+1}/{epochs} ==> Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

Epoch 1/10 ==> Train Loss: 0.1257, Val Loss: 0.0418
Epoch 2/10 ==> Train Loss: 0.0405, Val Loss: 0.0382
Epoch 3/10 ==> Train Loss: 0.0277, Val Loss: 0.0364
Epoch 4/10 ==> Train Loss: 0.0202, Val Loss: 0.0275
Epoch 5/10 ==> Train Loss: 0.0151, Val Loss: 0.0337
Epoch 6/10 ==> Train Loss: 0.0121, Val Loss: 0.0366
Epoch 7/10 ==> Train Loss: 0.0100, Val Loss: 0.0429
Epoch 8/10 ==> Train Loss: 0.0080, Val Loss: 0.0354
Epoch 9/10 ==> Train Loss: 0.0066, Val Loss: 0.0398
Epoch 10/10 ==> Train Loss: 0.0058, Val Loss: 0.0322


In [60]:
correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images.labels in testloader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)

        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)

print(f"accuracy={correct_labels/total_labels*100}")

accuracy=100.0
